# 基于MindSpore的GPT2模型新闻分类任务

## 环境准备

本案例的运行环境为：

| Python | MindSpore | MindSpore NLP |
| :----- | :-------- | :------------ |
| 3.10    | 2.7.0       | 0.5.1           |

如果你在如[昇思大模型平台](https://xihe.mindspore.cn/training-projects)、[华为云ModelArts](https://www.huaweicloud.com/product/modelarts.html)、[启智社区](https://openi.pcl.ac.cn/)等算力平台的Jupyter在线编程环境中运行本案例，可取消如下代码的注释，进行依赖库安装：

In [ ]:
# !pip install mindspore==2.7.0 mindnlp==0.5.1

### 数据集加载与处理

1. 数据集加载

    本次实验使用的是nlpcc2017摘要数据，内容为新闻正文及其摘要，总计50000个样本。

In [1]:
!rm -rf train_with_summ.txt
!wget https://download.mindspore.cn/toolkits/mindnlp/dataset/text_generation/nlpcc2017/train_with_summ.txt

--2025-12-15 10:17:48--  https://download.mindspore.cn/toolkits/mindnlp/dataset/text_generation/nlpcc2017/train_with_summ.txt
Resolving proxy-notebook.modelarts.com (proxy-notebook.modelarts.com)... 192.168.0.33
Connecting to proxy-notebook.modelarts.com (proxy-notebook.modelarts.com)|192.168.0.33|:8083... connected.
Proxy request sent, awaiting response... 200 OK
Length: 152337710 (145M) [application/octet-stream]
Saving to: ‘train_with_summ.txt’

train_with_summ.txt 100%[===================>] 145.28M  60.2MB/s    in 2.4s    

2025-12-15 10:17:50 (60.2 MB/s) - ‘train_with_summ.txt’ saved [152337710/152337710]



2. 数据预处理

    原始数据格式：
    ```text
    article: [CLS] article_context [SEP]
    summary: [CLS] summary_context [SEP]
    ```
    预处理后的数据格式：

    ```text
    [CLS] article_context [SEP] summary_context [SEP]
    ```

MindHF，MindSpore2.7.1

因GPT2无中文的tokenizer，我们使用BertTokenizer替代。

In [2]:
import mindnlp
from transformers import BertTokenizer

# We use BertTokenizer for tokenizing chinese context.
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')
len(tokenizer)

[WARNING] CORE(20623,ffff8c824640,python):2025-12-15-10:17:52.412.887 [mindspore/core/utils/ms_context.cc:533] GetJitLevel] Set jit level to O2 for rank table startup method.
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'nu

21128

In [3]:
import mindnlp
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
import numpy as np

class NLPCC2017Dataset(Dataset):
    def __init__(self, file_path, tokenizer, max_seq_len=1024, is_train=True):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.is_train = is_train
        self.samples = []
        
        # 读取数据文件
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line.strip())
                self.samples.append({
                    'article': data['article'],
                    'summarization': data['summarization']
                })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        article = sample['article']
        summary = sample['summarization']
        
        # 使用分词器处理文本
        tokenized = self.tokenizer(
            text=article,
            text_pair=summary if self.is_train else None,
            padding='max_length',
            truncation='only_first',
            max_length=self.max_seq_len,
            return_tensors='pt'
        )
        
        # 确保返回的字典包含所有必要的键
        result = {
            'input_ids': tokenized['input_ids'].squeeze(0),
            'labels': tokenized['input_ids'].squeeze(0),
            'attention_mask': tokenized['attention_mask'].squeeze(0)
        }
        
        return result

def create_dataloaders(data_path, tokenizer, batch_size=4, max_seq_len=1024, train_ratio=0.9):
    """
    创建训练和测试数据加载器
    
    Args:
        data_path: 数据文件路径
        tokenizer: 分词器
        batch_size: 批次大小
        max_seq_len: 最大序列长度
        train_ratio: 训练集比例
    """
    # 加载完整数据集
    full_dataset = NLPCC2017Dataset(data_path, tokenizer, max_seq_len)
    dataset_size = len(full_dataset)
    mini_size = int(0.001 * dataset_size)
    mini_dataset = torch.utils.data.Subset(full_dataset, range(0, mini_size))
    
    # 划分训练集和测试集
    dataset_size = len(mini_dataset)
    train_size = int(train_ratio * dataset_size)
    
    # train_dataset, test_dataset = torch.utils.data.split(
    #     mini_dataset, [train_size, test_size]
    # )
    
    train_dataset = torch.utils.data.Subset(mini_dataset, range(0, train_size))
    test_dataset = torch.utils.data.Subset(mini_dataset, range(train_size, dataset_size))
    
    # 设置训练集和测试集的模式
    train_dataset.dataset.is_train = True
    test_dataset.dataset.is_train = False
    
    # 创建数据加载器
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=False,  # 训练时打乱数据
        num_workers=0,
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=1,  # 测试时通常使用batch_size=1
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )
    
    return train_dataset, test_dataset, train_loader, test_loader



In [4]:
# 初始化分词器（与MindSpore版本保持一致）
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')

# 创建数据加载器
train_dataset, test_dataset, train_loader, test_loader = create_dataloaders(
    data_path='./train_with_summ.txt',  # 请替换为实际路径
    tokenizer=tokenizer,
    batch_size=1,
    max_seq_len=1024
)

# 测试数据加载器
print(f"训练集批次数量: {len(train_loader)}")
print(f"测试集批次数量: {len(test_loader)}")

# 查看一个批次的数据
for batch in train_loader:
    print("批次数据形状:")
    print(f"input_ids: {batch['input_ids'].shape}")
    print(f"input_ids: {batch['input_ids']}")
    print(f"attention_mask: {batch['attention_mask'].shape}")
    print(f"labels: {batch['labels'].shape}")
    break

训练集批次数量: 45
测试集批次数量: 5
[MS_ALLOC_CONF] config:  enable_vmm:True  vmm_align_size:2MB
批次数据形状:
input_ids: mindtorch.Size([1, 1024])
input_ids: [[ 101 1724 3862 ...    0    0    0]]
attention_mask: mindtorch.Size([1, 1024])
labels: mindtorch.Size([1, 1024])


### 模型构建

1. 构建GPT2ForSummarization模型，注意***shift right***的操作。

In [5]:
from mindspore.mint.nn import functional as F
from transformers import GPT2LMHeadModel

class GPT2ForSummarization(GPT2LMHeadModel):
    def forward(
        self,
        input_ids = None,
        attention_mask = None,
        labels = None,
    ):
        outputs = super().forward(input_ids=input_ids, attention_mask=attention_mask)
        shift_logits = outputs.logits[..., :-1, :]
        shift_labels = labels[..., 1:]
        # Flatten the tokens
        loss = F.cross_entropy(shift_logits.view(-1, shift_logits.shape[-1]), shift_labels.view(-1), ignore_index=tokenizer.pad_token_id)
        return (loss,)

### 模型训练

In [7]:
num_epochs = 1
warmup_steps = 100
learning_rate = 1.5e-4
max_grad_norm = 1.0
num_training_steps = num_epochs * len(train_dataset)

In [8]:
from mindspore import nn
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(vocab_size=len(tokenizer))
model = GPT2ForSummarization(config)

In [9]:
# 记录模型参数数量
print('number of model parameters: {}'.format(model.num_parameters()))

number of model parameters: 102068736


In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="gpt2_summarization",
    save_steps=len(train_dataset),
    save_total_limit=3,
    logging_steps=1000,
    max_steps=num_training_steps,
    learning_rate=learning_rate,
    max_grad_norm=max_grad_norm,
    warmup_steps=warmup_steps
    
)

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

Detected kernel version 4.19.90, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [12]:
trainer.train()

.....

Step,Training Loss


TrainOutput(global_step=45, training_loss=8.821623399522569, metrics={'train_runtime': 95.8106, 'train_samples_per_second': 3.757, 'train_steps_per_second': 0.47, 'total_flos': 177155997696000.0, 'train_loss': 8.821623399522569, 'epoch': 7.5})

In [13]:
# 查看一个批次的数据
for batch in test_loader:
    print("批次数据形状:")
    print(f"input_ids: {batch['input_ids'].shape}")
    print(f"input_ids: {batch['input_ids']}")
    print(f"attention_mask: {batch['attention_mask'].shape}")
    print(f"labels: {batch['labels'].shape}")
    break

批次数据形状:
input_ids: mindtorch.Size([1, 1024])
input_ids: [[ 101 4373 3360 ...    0    0    0]]
attention_mask: mindtorch.Size([1, 1024])
labels: mindtorch.Size([1, 1024])


In [14]:
for batch in test_loader:
    print(f"input_ids: {batch['input_ids']}")
    print(tokenizer.decode(batch['input_ids'][0]))
    break

input_ids: [[ 101 4373 3360 ...    0    0    0]]
[CLS] 玉 林 新 闻 网 - 玉 林 晚 报 讯 （ 通 讯 员 < [UNK] > 黄 传 庆 ） 练 车 间 隙 ， 发 现 训 练 场 附 近 一 根 电 线 杆 上 有 一 个 鸟 窝 ， 21 岁 的 刁 某 立 即 爬 上 去 捉 鸟 ， 不 想 就 此 命 丧 黄 泉 。 日 前 发 生 在 博 白 县 文 地 镇 一 驾 校 训 练 场 的 意 外 事 故 ， 令 人 唏 嘘 。 今 年 21 岁 的 刁 某 是 博 白 县 宁 潭 镇 新 荣 村 人 ， 不 久 前 到 其 堂 叔 刁 某 某 任 教 练 的 宁 潭 镇 某 驾 校 参 加 汽 车 驾 驶 员 培 训 。 25 日 中 午 ， 刁 某 随 堂 叔 刁 某 某 及 另 外 几 名 学 员 开 车 到 文 地 镇 ， 借 用 文 地 镇 钛 白 粉 厂 附 近 的 训 练 场 练 车 。 14 时 许 ， 训 练 间 隙 ， 刁 某 发 现 训 练 场 附 近 一 根 电 线 杆 上 有 一 个 鸟 窝 ， 一 只 [UNK] 八 哥 [UNK] 正 叼 着 虫 子 飞 回 鸟 窝 。 平 时 刁 某 偶 尔 会 捕 鸟 出 卖 ， 知 道 这 种 鸟 价 值 100 多 元 。 他 跟 身 边 学 员 打 了 个 招 呼 ， 便 飞 奔 过 去 ， 欲 爬 上 电 线 杆 捉 鸟 。 刁 某 某 发 现 后 追 过 去 欲 阻 止 ， 但 为 时 已 晚 ， 等 刁 某 某 赶 到 时 ， 刁 某 已 爬 到 电 线 杆 顶 ， 伸 手 捉 鸟 时 不 慎 触 碰 到 头 顶 的 高 压 电 线 ， 当 即 身 亡 。 其 手 掌 被 外 露 的 钢 枝 刺 穿 ， 尸 体 悬 挂 在 电 线 杆 上 。 26 日 下 午 ， 在 文 地 、 宁 潭 镇 政 府 及 相 关 部 门 协 调 下 ， 相 关 责 任 方 与 死 者 家 属 达 成 赔 偿 协 议 ， 死 者 家 属 同 意 将 死 者 尸 体 取 下 搬 走 ， 相 关 责 任 方 共 赔 偿 死 者 家 属 15. 8 万 元 ， 其 中 宁 潭 镇 某 驾 校 赔 付 7. 3 万 元 ， 死 者 堂 叔 刁 某

In [15]:
model = GPT2LMHeadModel.from_pretrained('./gpt2_summarization/checkpoint-45', config=config)

由于训练数据量少，epochs数少且tokenizer并未使用gpt tokenizer等因素，模型推理效果会较差。

In [16]:
model.set_train(False)
model.config.eos_token_id = model.config.sep_token_id
i = 0

for batch in test_loader:
    output_ids = model.generate(batch['input_ids'], max_new_tokens=50, num_beams=5, no_repeat_ngram_size=2)
    output_text = tokenizer.decode(output_ids[0].tolist())
    print(output_text)
    i += 1
    if i == 1:
        break

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (1024). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


[CLS] 玉 林 新 闻 网 - 玉 林 晚 报 讯 （ 通 讯 员 < [UNK] > 黄 传 庆 ） 练 车 间 隙 ， 发 现 训 练 场 附 近 一 根 电 线 杆 上 有 一 个 鸟 窝 ， 21 岁 的 刁 某 立 即 爬 上 去 捉 鸟 ， 不 想 就 此 命 丧 黄 泉 。 日 前 发 生 在 博 白 县 文 地 镇 一 驾 校 训 练 场 的 意 外 事 故 ， 令 人 唏 嘘 。 今 年 21 岁 的 刁 某 是 博 白 县 宁 潭 镇 新 荣 村 人 ， 不 久 前 到 其 堂 叔 刁 某 某 任 教 练 的 宁 潭 镇 某 驾 校 参 加 汽 车 驾 驶 员 培 训 。 25 日 中 午 ， 刁 某 随 堂 叔 刁 某 某 及 另 外 几 名 学 员 开 车 到 文 地 镇 ， 借 用 文 地 镇 钛 白 粉 厂 附 近 的 训 练 场 练 车 。 14 时 许 ， 训 练 间 隙 ， 刁 某 发 现 训 练 场 附 近 一 根 电 线 杆 上 有 一 个 鸟 窝 ， 一 只 [UNK] 八 哥 [UNK] 正 叼 着 虫 子 飞 回 鸟 窝 。 平 时 刁 某 偶 尔 会 捕 鸟 出 卖 ， 知 道 这 种 鸟 价 值 100 多 元 。 他 跟 身 边 学 员 打 了 个 招 呼 ， 便 飞 奔 过 去 ， 欲 爬 上 电 线 杆 捉 鸟 。 刁 某 某 发 现 后 追 过 去 欲 阻 止 ， 但 为 时 已 晚 ， 等 刁 某 某 赶 到 时 ， 刁 某 已 爬 到 电 线 杆 顶 ， 伸 手 捉 鸟 时 不 慎 触 碰 到 头 顶 的 高 压 电 线 ， 当 即 身 亡 。 其 手 掌 被 外 露 的 钢 枝 刺 穿 ， 尸 体 悬 挂 在 电 线 杆 上 。 26 日 下 午 ， 在 文 地 、 宁 潭 镇 政 府 及 相 关 部 门 协 调 下 ， 相 关 责 任 方 与 死 者 家 属 达 成 赔 偿 协 议 ， 死 者 家 属 同 意 将 死 者 尸 体 取 下 搬 走 ， 相 关 责 任 方 共 赔 偿 死 者 家 属 15. 8 万 元 ， 其 中 宁 潭 镇 某 驾 校 赔 付 7. 3 万 元 ， 死 者 堂 叔 刁 某 某 （ 驾 校 教 练 ） 赔 付 5. 3 万 元 ， 文 地 供 电 所 本 来 没 有 直